In [ ]:
#Install and load necessary packages
install.packages(c("foreach", "doParallel", "MASS"))
library(foreach)
library(doParallel)
#library(tictoc) # for timing the simulation

# Define the core simulation function


# --- Helpers ---------------------------------------------------------------
gen_sameCV_oppositeSkew_MC2 <- function(n,
                                        mean1, mean2,
                                        CV,
                                        rho_latent = 0.6,
                                        # If k_weibull is NULL, solve k from CV
                                        k_weibull = NULL,
                                        seed = NULL) {
  if (!is.null(seed)) set.seed(seed)
  if (mean1 <= 0 || mean2 <= 0) stop("This setup assumes positive means (CV = sd/mean).")
  if (CV <= 0) stop("CV must be > 0.")

  # Weibull CV depends only on shape k
  weibull_cv <- function(k) {
    g1 <- gamma(1 + 1/k)
    g2 <- gamma(1 + 2/k)
    sqrt(g2 / (g1^2) - 1)
  }

  # Solve k from target CV if not provided
  if (is.null(k_weibull)) {
    f <- function(k) weibull_cv(k) - CV
    k_weibull <- uniroot(f, interval = c(0.1, 50))$root
  }

  # 1) Gaussian copula uniforms
  S <- matrix(c(1, rho_latent, rho_latent, 1), 2, 2)
  Z <- MASS::mvrnorm(n, mu = c(0, 0), Sigma = S)
  U <- pnorm(Z[, 1])
  V <- pnorm(Z[, 2])

  # 2) X: right-skewed Weibull with population mean = mean1
  lambda1 <- mean1 / gamma(1 + 1/k_weibull)
  X <- qweibull(U, shape = k_weibull, scale = lambda1)

  # 3) Build Y as a LEFT-skewed mirror of a right-skewed Weibull:
  #    Use Vtilde = 1 - V so Y stays increasing in Z2 (keeps positive dependence)
  Vtilde <- 1 - V

  # Start from a right-skewed Weibull "T" with any positive scale (we standardize anyway)
  T_raw <- qweibull(Vtilde, shape = k_weibull, scale = 1)

  # Standardize using THEORETICAL mean/sd of Weibull(k, scale=1)
  muT <- gamma(1 + 1/k_weibull)
  sdT <- sqrt(gamma(1 + 2/k_weibull) - muT^2)

  sd2_tgt <- CV * mean2

  T_std <- (T_raw - muT) / sdT * sd2_tgt   # mean 0, sd = sd2_tgt (population-level)
  Y <- mean2 - T_std                        # mirror -> left skew (negative skewness)

  list(
    X = X,
    Y = Y,
    summary = list(
      mean_X = mean(X), sd_X = sd(X), CV_X = sd(X)/mean(X),
      mean_Y = mean(Y), sd_Y = sd(Y), CV_Y = sd(Y)/mean(Y),
      cor_Pearson = cor(X, Y),
      cor_Spearman = cor(X, Y, method = "spearman"),
      params = list(k_weibull = k_weibull, lambda1 = lambda1, rho_latent = rho_latent)
    )
  )
}



# Profiled negative log-likelihood under H0 (shared m with delta shift)
nll0_profile <- function(par, x, y, delta) {
  lsX <- par[1]; lsY <- par[2]; zr <- par[3]
  sX  <- exp(lsX)
  sY  <- exp(lsY)
  rho <- tanh(zr)

  # profiled m-hat given (sX,sY) and delta
  mhat <- 0.5 * (mean(x)/sX + mean(y)/sY - delta)

  # standardized residuals
  rx <- x/sX - mhat
  ry <- y/sY - (mhat + delta)

  one_m_r2 <- 1 - rho^2
  n <- length(x)

  quad <- sum(rx^2 - 2*rho*rx*ry + ry^2) / one_m_r2
  logdet <- 2*log(sX) + 2*log(sY) + log(one_m_r2)

  0.5 * (n*log(2*pi) + n*logdet + quad)
}

# Profiled negative log-likelihood under H1 (separate means)
nll1_profile <- function(par, x, y) {
  lsX <- par[1]; lsY <- par[2]; zr <- par[3]
  sX  <- exp(lsX)
  sY  <- exp(lsY)
  rho <- tanh(zr)

  # profiled mX-hat, mY-hat
  mXhat <- mean(x)/sX
  mYhat <- mean(y)/sY

  rx <- x/sX - mXhat
  ry <- y/sY - mYhat

  one_m_r2 <- 1 - rho^2
  n <- length(x)

  quad <- sum(rx^2 - 2*rho*rx*ry + ry^2) / one_m_r2
  logdet <- 2*log(sX) + 2*log(sY) + log(one_m_r2)

  0.5 * (n*log(2*pi) + n*logdet + quad)
}




#get_LRT_CI <- function(delta, LRT, crit, d_by) {
#  keep <- delta[LRT < crit]
#  if (length(keep) == 0) return(c(NA, NA))
#  range(keep) + c(-d_by, d_by)
#}




#Part 2: Simmulation ###################################################################################

# 2. Set up the parallel backend
# Detect the number of available cores and use them (leaving one free for system tasks)
n_cores <- detectCores()
if (n_cores > 1) {
  # Create a cluster
  cl <- makeCluster(n_cores - 1)
  registerDoParallel(cl)
  cat(paste("Registered parallel backend with", n_cores - 1, "cores.\n"))
} else {
  registerDoSEQ() # Fallback to sequential if only one core is available
  cat("Only one core detected, running sequentially.\n")
}

# 3. Run the long simulation in parallel

nsim <- 1000 # A large number for a long simulation
true_delta <- 0   # Model 1: eta_Y - eta_X = 0
cat(paste("Starting", nsim, "simulations...\n"))

n_sample_size=c(30, 100, 500, 1000) #Sample Size
rho_correlation=c(-.7, -.3, 0, .3, .7) #Correlation
#nsim=2 #Number of Simulation

Out_Matrix<-matrix(NA, nrow=20, ncol=6)


t0<-proc.time() # Start timing

for(i1 in 1:length(rho_correlation)){
    for(i2 in 1:length(n_sample_size)){


# The %dopar% operator tells foreach to run the iterations in parallel
results <- foreach(kk = 1:nsim, .combine = rbind) %dopar% {


n<-n_sample_size[i2]

set <- gen_sameCV_oppositeSkew_MC2(n = n_sample_size[i2],
                                   mean1 = 5, mean2 = 8,
                                   CV = 0.4, rho_latent = rho_correlation[i1],
                                   seed = (2025 + kk))

xt<-set$X
yt<-set$Y


# good starting values from sample moments
    init_sX  <- sd(xt)
    init_sY  <- sd(yt)
    init_rho <- cor(xt, yt)
    par0 <- c(log(init_sX), log(init_sY), atanh(pmax(pmin(init_rho, 0.99), -0.99)))

    crit <- qchisq(.95, 1)
    true_delta <- 0   # Model 1: eta_Y - eta_X = 0

    # H1:
    op1 <- optim(par0, nll1_profile, x = xt, y = yt, method = "Nelder-Mead",
                 control = list(maxit = 4000, reltol = 1e-10))

    # LRT(delta) - crit; if it fails: NA
    g <- function(d) {
      op0 <- optim(par0, nll0_profile, x = xt, y = yt, delta = d, method = "Nelder-Mead",
                   control = list(maxit = 4000, reltol = 1e-10))
      if (op0$convergence != 0) return(NA)
      2 * (op0$value - op1$value) - crit
    }

    # MLE of delta (= eta_Y - eta_X)
    sXm <- sqrt(mean((xt - mean(xt))^2)); sYm <- sqrt(mean((yt - mean(yt))^2))
    d_hat <- mean(yt)/sYm - mean(xt)/sXm

    find_end <- function(dir, max_step = 2^10) {
      step <- 1
      repeat {
        b  <- d_hat + dir * step
        gb <- g(b)
        if (is.na(gb)) return(NA)
        if (gb > 0) break
        step <- 2 * step
        if (step > max_step) return(NA)
      }
      uniroot(g, sort(c(d_hat, b)), tol = 1e-6)$root
    }

    Lower_CI <- find_end(-1)
    Upper_CI <- find_end(+1)

    if (op1$convergence != 0 || is.na(Lower_CI) || is.na(Upper_CI)) {
      data.frame(sim_id = kk, Cov.out = NA, Len_out = NA,
                 Lower = NA, Upper = NA, fail = 1)
    } else {
      data.frame(sim_id = kk,
                 Cov.out = as.numeric(Lower_CI <= true_delta & true_delta <= Upper_CI),
                 Len_out = Upper_CI - Lower_CI,
                 Lower = Lower_CI, Upper = Upper_CI, fail = 0)
    }
} # End of foreach

        Cov_rate_out<-results[,2]
        Length_CI_out<-results[,3]
        Cov_rate_out[is.infinite(Cov_rate_out)]<-NA
        Length_CI_out[is.infinite(Length_CI_out)]<-NA
        Cov_rate_out<-na.omit(Cov_rate_out)
        Length_CI_out<-na.omit(Length_CI_out)


        ok <- results$fail == 0                      # only successes
        cov_i  <- results$Cov.out[ok]
        len_i  <- results$Len_out[ok]
        low_i  <- results$Lower[ok]
        up_i   <- results$Upper[ok]
        R      <- sum(ok)                            # number of "s"

        OM <- (i1 - 1) * length(n_sample_size) + i2
       Out_Matrix[OM, 1] <- mean(cov_i)                              # coverage
       Out_Matrix[OM, 2] <- round(mean(len_i), 4)                    # ave. length
       Out_Matrix[OM, 3] <- sum(results$fail)                        # number of "f"
       Out_Matrix[OM, 4] <- round(sqrt(mean(cov_i) * (1 - mean(cov_i)) / R), 4)  # coverage의 MC SE
       Out_Matrix[OM, 5] <- mean(true_delta < low_i)                 # Lower miss
       Out_Matrix[OM, 6] <- mean(true_delta > up_i)                  # Upper miss

    }
}



# 4. Stop the cluster and analyze results
if (exists("cl")) {
  stopCluster(cl) # Important: always stop the cluster when done
}


t1<-proc.time() # Stop timing

t1-t0
Out_Matrix


Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependency ‘iterators’


Loading required package: iterators

Loading required package: parallel



Registered parallel backend with 7 cores.
Starting 1000 simulations...


   user  system elapsed 
 13.923   1.984 381.200 

0.951,1.9015,0,0.0068,0.023,0.026
0.954,1.0183,0,0.0066,0.023,0.023
0.949,0.4510,0,0.0070,0.026,0.025
0.953,0.3189,0,0.0067,0.023,0.024
0.953,2.1159,0,0.0067,0.021,0.026
0.955,1.1385,0,0.0066,0.025,0.020
0.950,0.5054,0,0.0069,0.023,0.027
0.963,0.3573,0,0.0060,0.018,0.019
0.957,2.1055,0,0.0064,0.023,0.020
0.951,1.1346,0,0.0068,0.027,0.022
0.956,0.5044,0,0.0065,0.017,0.027
